## Bedrock Depth â€” Data Cleaning

In [24]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

df = pd.read_csv(r"C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\bedrock_depth.csv")
print(f"Raw rows: {len(df)}")
print(df.dtypes)
df.head()

Raw rows: 292
fid            int64
source        object
data_type     object
elev_ft      float64
depth_ft     float64
latitude     float64
longitude    float64
dtype: object


,fid,source,data_type,elev_ft,depth_ft,latitude,longitude
0,264,Map or Cross-Section,Water/Monitoring Well,58.2790,42.0,40.915556,-73.909722
1,265,Map or Cross-Section,Water/Monitoring Well,191.9300,50.0,40.918889,-73.866111
2,266,Map or Cross-Section,Water/Monitoring Well,34.7776,9.0,40.894444,-73.791111
3,267,Map or Cross-Section,Water/Monitoring Well,66.1750,34.0,40.910000,-73.781944
4,268,Map or Cross-Section,Water/Monitoring Well,-3.5749,60.0,40.919444,-73.787500


### Step 1 â€” Remove rows with invalid depth (depth_ft <= 0)

In [25]:
before = len(df)
df = df[df["depth_ft"] > 0].reset_index(drop=True)
print(f"Removed {before - len(df)} rows with depth_ft <= 0 | Remaining: {len(df)}")

Removed 1 rows with depth_ft <= 0 | Remaining: 291


### Step 2 â€” Remove duplicate lat/lon pairs (keep first occurrence)

In [26]:
before = len(df)
df = df.drop_duplicates(subset=["latitude", "longitude"], keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate lat/lon rows | Remaining: {len(df)}")

Removed 16 duplicate lat/lon rows | Remaining: 275


### Step 3 â€” Remove points outside NYC boundary

In [27]:
# Load NYC boundary and reproject to WGS84 for coordinate-level filtering
nyc = gpd.read_file(r"C:\Users\szaha\Downloads\Borough_Boundaries_20260328.geojson")
nyc_union = nyc.dissolve().to_crs(epsg=4326).reset_index(drop=True)

# Convert bedrock data to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

before = len(gdf)
gdf = gdf[gdf.within(nyc_union.union_all())].reset_index(drop=True)
print(f"Removed {before - len(gdf)} points outside NYC boundary | Remaining: {len(gdf)}")

Removed 89 points outside NYC boundary | Remaining: 186


### Step 4 â€” Standardize column names and drop geometry column

In [28]:
# Standardize to lowercase column names (consistent with other datasets)
df_clean = gdf.drop(columns=["geometry"]).rename(columns={
    "fid":       "fid",
    "source":    "source",
    "data_type": "data_type",
    "elev_ft":   "elev_ft",
    "depth_ft":  "depth_ft",
    "latitude":  "latitude",
    "longitude": "longitude"
})

# Round numeric columns to 4 decimal places
df_clean["elev_ft"]   = df_clean["elev_ft"].round(4)
df_clean["depth_ft"]  = df_clean["depth_ft"].round(4)
df_clean["latitude"]  = df_clean["latitude"].round(6)
df_clean["longitude"] = df_clean["longitude"].round(6)

print(f"Final clean shape: {df_clean.shape}")
print(df_clean.dtypes)
df_clean.head()

Final clean shape: (186, 7)
fid            int64
source        object
data_type     object
elev_ft      float64
depth_ft     float64
latitude     float64
longitude    float64
dtype: object


,fid,source,data_type,elev_ft,depth_ft,latitude,longitude
0,15,WPA,Construction Borehole,-170.0,183.66,40.713611,-73.952222
1,16,WPA,Construction Borehole,-133.0,152.22,40.710556,-73.954167
2,0,USGS/NWIS,Water/Monitoring Well,-356.0,401.00,40.638333,-73.979722
3,1,USGS/NWIS,Water/Monitoring Well,-387.0,429.29,40.631667,-73.970278
4,3,USGS/NWIS,Water/Monitoring Well,-573.0,578.82,40.578056,-73.988056


### Step 5 â€” Final checks

In [29]:
print("Null counts:")
print(df_clean.isnull().sum())
print()
print("Numeric ranges:")
print(df_clean[["elev_ft", "depth_ft", "latitude", "longitude"]].describe())
print()
print("Unique sources:", df_clean["source"].unique())
print("Unique data types:", df_clean["data_type"].unique())

Null counts:
fid          0
source       0
data_type    0
elev_ft      0
depth_ft     0
latitude     0
longitude    0
dtype: int64

Numeric ranges:
          elev_ft    depth_ft    latitude   longitude
count  186.000000  186.000000  186.000000  186.000000
mean  -189.572166  231.715983   40.718174  -73.927409
std    192.290415  187.788648    0.059507    0.081768
min   -974.000000    3.040000   40.543056  -74.198611
25%   -247.250000  101.302500   40.688611  -73.965764
50%   -141.500000  176.510000   40.728472  -73.928750
75%    -66.250000  295.622500   40.750417  -73.887292
max    253.950000  979.990000   40.894167  -73.734167

Unique sources: ['WPA' 'USGS/NWIS' 'USGS/NWIS/GeoLog' 'USGS']
Unique data types: ['Construction Borehole' 'Water/Monitoring Well']


### Step 6 â€” Save cleaned output

In [30]:
output_path = r"C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\cleaned_bedrock.csv"
df_clean.to_csv(output_path, index=False)
print(f"Saved {len(df_clean)} rows to {output_path}")

Saved 186 rows to C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\cleaned_bedrock.csv


---
## Pavement Ratings — Data Cleaning

In [31]:
import pandas as pd
import geopandas as gpd

df_pave = pd.read_csv(r"C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\pavement_ratings.csv")
print(f"Raw rows: {len(df_pave):,}")
print(df_pave.dtypes)
df_pave.head()

Raw rows: 488,228
borough             object
street_name         object
road_type           object
system_rating      float64
inspection_date     object
latitude           float64
longitude          float64
dtype: object


,borough,street_name,road_type,system_rating,inspection_date,latitude,longitude
0,Manhattan,1 AVENUE,NaN,9.11,04/30/2022 12:00:00 AM,40.734352,-73.980230
1,Manhattan,1 AVENUE,NaN,8.68,06/16/2018 12:00:00 AM,40.733197,-73.981070
2,Manhattan,1 AVENUE,NaN,5.83,06/02/2018 12:00:00 AM,40.733197,-73.981070
3,Manhattan,1 AVENUE,NaN,8.62,07/06/2019 12:00:00 AM,40.734631,-73.980028
4,Manhattan,1 AVENUE,NaN,9.19,12/04/2021 12:00:00 AM,40.734631,-73.980028


### Step 1 — Drop rows with null borough or street_name

In [32]:
before = len(df_pave)
df_pave = df_pave.dropna(subset=["borough", "street_name"])
df_pave = df_pave[df_pave["borough"].str.strip() != ""]
df_pave = df_pave[df_pave["street_name"].str.strip() != ""]
df_pave = df_pave.reset_index(drop=True)
print(f"Removed {before - len(df_pave):,} rows with null borough/street_name | Remaining: {len(df_pave):,}")

Removed 89 rows with null borough/street_name | Remaining: 488,139


### Step 2 — Parse inspection_date and add year_month column

In [33]:
df_pave["inspection_date"] = pd.to_datetime(df_pave["inspection_date"], format="%m/%d/%Y %I:%M:%S %p")
df_pave["year_month"] = df_pave["inspection_date"].dt.to_period("M").astype(str)
print(f"Date range: {df_pave['inspection_date'].min().date()} to {df_pave['inspection_date'].max().date()}")
df_pave[["inspection_date", "year_month"]].head()

Date range: 2006-10-20 to 2026-02-23


,inspection_date,year_month
0,2022-04-30,2022-04
1,2018-06-16,2018-06
2,2018-06-02,2018-06
3,2019-07-06,2019-07
4,2021-12-04,2021-12


### Step 3 — Clip to common date range (2020-01 to 2025-11)

In [34]:
before = len(df_pave)
df_pave = df_pave[
    (df_pave["year_month"] >= "2020-01") &
    (df_pave["year_month"] <= "2025-11")
].reset_index(drop=True)
print(f"Removed {before - len(df_pave):,} rows outside date range | Remaining: {len(df_pave):,}")
print(f"Date range after clip: {df_pave['year_month'].min()} to {df_pave['year_month'].max()}")

Removed 139,686 rows outside date range | Remaining: 348,453
Date range after clip: 2020-01 to 2025-11


### Step 4 — Remove duplicate (lat, lon, inspection_date) rows

In [35]:
# Same location + same date = true duplicate inspection record
# (duplicate lat/lon alone is expected — same road inspected across different months)
before = len(df_pave)
df_pave = df_pave.drop_duplicates(
    subset=["latitude", "longitude", "inspection_date"],
    keep="first"
).reset_index(drop=True)
print(f"Removed {before - len(df_pave):,} duplicate (lat, lon, date) rows | Remaining: {len(df_pave):,}")

Removed 27,886 duplicate (lat, lon, date) rows | Remaining: 320,567


### Step 5 — Remove points outside NYC boundary

In [36]:
nyc = gpd.read_file(r"C:\Users\szaha\Downloads\Borough_Boundaries_20260328.geojson")
nyc_union = nyc.dissolve().to_crs(epsg=4326).reset_index(drop=True)

gdf_pave = gpd.GeoDataFrame(
    df_pave,
    geometry=gpd.points_from_xy(df_pave["longitude"], df_pave["latitude"]),
    crs="EPSG:4326"
)

before = len(gdf_pave)
gdf_pave = gdf_pave[gdf_pave.within(nyc_union.union_all())].reset_index(drop=True)
print(f"Removed {before - len(gdf_pave):,} points outside NYC boundary | Remaining: {len(gdf_pave):,}")

Removed 85 points outside NYC boundary | Remaining: 320,482


### Step 6 — Standardize columns

In [37]:
df_clean_pave = gdf_pave.drop(columns=["geometry"]).copy()

# Standardize borough to uppercase (matches other datasets)
df_clean_pave["borough"] = df_clean_pave["borough"].str.upper().str.strip()

# Standardize street_name
df_clean_pave["street_name"] = df_clean_pave["street_name"].str.strip()

# Round numeric columns
df_clean_pave["latitude"]      = df_clean_pave["latitude"].round(6)
df_clean_pave["longitude"]     = df_clean_pave["longitude"].round(6)
df_clean_pave["system_rating"] = df_clean_pave["system_rating"].round(4)

# Final column order
df_clean_pave = df_clean_pave[["borough", "street_name", "road_type",
                                "system_rating", "inspection_date",
                                "year_month", "latitude", "longitude"]]

print(f"Final clean shape: {df_clean_pave.shape}")
print(df_clean_pave.dtypes)
df_clean_pave.head()

Final clean shape: (320482, 8)
borough                    object
street_name                object
road_type                  object
system_rating             float64
inspection_date    datetime64[ns]
year_month                 object
latitude                  float64
longitude                 float64
dtype: object


,borough,street_name,road_type,system_rating,inspection_date,year_month,latitude,longitude
0,MANHATTAN,1 AVENUE,NaN,9.11,2022-04-30,2022-04,40.734352,-73.980230
1,MANHATTAN,1 AVENUE,NaN,9.19,2021-12-04,2021-12,40.734631,-73.980028
2,MANHATTAN,1 AVENUE,NaN,9.83,2025-11-09,2025-11,40.734631,-73.980028
3,MANHATTAN,1 AVENUE,NaN,9.11,2023-01-07,2023-01,40.734631,-73.980028
4,MANHATTAN,1 AVENUE,NaN,8.57,2024-11-09,2024-11,40.734631,-73.980028


### Step 7 — Final checks

In [38]:
print("Null counts:")
print(df_clean_pave.isnull().sum())
print()
print("Numeric ranges:")
print(df_clean_pave[["system_rating", "latitude", "longitude"]].describe())
print()
print("Unique boroughs:", sorted(df_clean_pave["borough"].unique()))
print("year_month range:", df_clean_pave["year_month"].min(), "->", df_clean_pave["year_month"].max())

Null counts:
borough                 0
street_name             0
road_type          301779
system_rating           0
inspection_date         0
year_month              0
latitude                0
longitude               0
dtype: int64

Numeric ranges:
       system_rating       latitude      longitude
count  320482.000000  320482.000000  320482.000000
mean        7.074952      40.703011     -73.915645
std         3.100440       0.087101       0.118242
min         0.000000      40.497961     -74.254946
25%         5.810000      40.636084     -73.978111
50%         7.960000      40.700516     -73.905698
75%         9.650000      40.757961     -73.827315
max        10.000000      40.912351     -73.700508

Unique boroughs: ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
year_month range: 2020-01 -> 2025-11


### Step 8 — Save cleaned output

In [39]:
output_path = r"C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\cleaned_pavement.csv"
df_clean_pave.to_csv(output_path, index=False)
print(f"Saved {len(df_clean_pave):,} rows to {output_path}")

Saved 320,482 rows to C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f\cleaned_pavement.csv
